# Verifying the Adapt Version

Adapt needs to be on verison 0.1.3.post to run the postprocessing. If adapt --version shows 0.1.3, run the following commands in terminal

    Git clone https://github.com/ARM-DOE/Adapt.git

    cd Adapt

    git fetch origin 

    git switch wip

    git branch #to confirm wip branch is active

    pip install -e

From there adapt --version should show adapt 0.1.3post and the notebook can be run. 

In [1]:
#Imports
import glob
import numpy as np
import datetime
import xarray as xr
import pandas as pd
import pyproj as proj4
import sqlite3

from pyxlma.lmalib.io import read as lma_read
from pyxlma.lmalib.flash.cluster import cluster_flashes
from pyxlma.lmalib.flash.properties import flash_stats, filter_flashes
from pyxlma.lmalib.grid import  create_regular_grid, assign_regular_bins, events_to_grid
from pyxlma.plot.xlma_plot_feature import color_by_time, plot_points, setup_hist, plot_3d_grid, subset
from pyxlma.plot.xlma_base_plot import subplot_labels, inset_view, BlankPlot

import sys, glob



# Process Lightning Data

The example here uses a case from [North Alabama](https://lma-tech.com/nalma/), but lightning networks exist in [North Colorado](http://lightning.nmt.edu/colma/), [Houston](https://lightning.geos.tamu.edu/hlma/), [New Mexico](http://www.lightning.nmt.edu/ll_lma/), [Oklahoma](https://apps.nssl.noaa.gov/oklma/current.php), [West Texas](http://lightning.ttu.edu/wtlma/), and others. Each location includes a link to the online archives for case selection and real-time forecasting. 

The [North Alabama Data](https://search.earthdata.nasa.gov/search?q=LMA) and [Oklahoma Data](https://data.nssl.noaa.gov/thredds/catalog/WRDD/OKLMA/catalog.html) can be found at the embedded links. Most other data is available upon request to the opperating organisations of each network.

This first section of the notebook will show how to process lightning data and add additional datasets to the full dataset.

In [2]:
#Configs

# Adjust this to match the length of the dataset we read in. It is used to set up 
# the total duration of the gridding along the time dimension.
duration_min = 60*1 + 10

# Source to flash 
chi2max = 1.0
stationsmin = 6
min_events_per_flash = 5

# There is a parameter to change the gridding from the 1 min default time resolution.
grid_time_delta_sec = 60*5
resolution_m = 4000
latlon_grid=False # False uses the azimuthal equidistant projection coordinate grid.

In [3]:
#Make aa list of lma files to be processed

#Change the directory to match LMA file list
filenames = glob.glob('/*/*/Adapt_pyxlma/Lightning_Data/NALMA_210504_1[3-4]*_0600.dat.gz')
#filenames += glob.glob('/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_YYMMDD_HHMMSS_0600.dat.gz')

filenames.sort()
print(len(filenames), "Total Files")
for filename in filenames:
    print(filename)

print("Reading files")
lma_data, starttime = lma_read.dataset(filenames)

good_events = (lma_data.event_stations >= stationsmin) & (lma_data.event_chi2 <= chi2max)
lma_data = lma_data[{'number_of_events':good_events}]

dttuple = [starttime, starttime+datetime.timedelta(minutes=duration_min)]
# dttuple = lma_data.Datetime.min(), lma_data.Datetime.max()
tstring = 'LMA {}-{}'.format(dttuple[0].strftime('%H%M'),
                                  dttuple[1].strftime('%H%M UTC %d %B %Y '))
print(tstring)

print("Clustering flashes")
ds = cluster_flashes(lma_data)
print("Calculating flash stats")
ds = flash_stats(ds)
ds = filter_flashes(ds, flash_event_count=(min_events_per_flash, None))

#Show the dataset to make sure there are flashes in the files
ds

7 Total Files
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_130000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_131000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_132000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_133000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_134000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_135000_0600.dat.gz
/home/blamsma/Adapt_pyxlma/Lightning_Data/NALMA_210504_140000_0600.dat.gz
Reading files
LMA 1300-1410 UTC 04 May 2021 
Clustering flashes
Calculating flash stats


<xarray.Dataset> Size: 157MB
Dimensions:                              (number_of_stations: 14,
                                          number_of_events: 1874138,
                                          number_of_flashes: 18600)
Coordinates:
  * number_of_flashes                    (number_of_flashes) uint64 149kB 0 ....
Dimensions without coordinates: number_of_stations, number_of_events
Data variables: (12/42)
    network_center_latitude              float64 8B 34.72
    network_center_longitude             float64 8B -86.65
    network_center_altitude              float64 8B 0.0
    station_latitude                     (number_of_stations) float32 56B 34....
    station_longitude                    (number_of_stations) float32 56B -87...
    station_altitude                     (number_of_stations) float32 56B 207...
    ...                                   ...
    flash_event_count                    (number_of_flashes) uint32 74kB 6 .....
    event_parent_flash_id                (number_of_events) uint64 15MB 0 ......
    event_x                              (number_of_events) float64 15MB -1.2...
    event_y                              (number_of_events) float64 15MB -4.6...
    event_z                              (number_of_events) float32 7MB 1.009...
    flash_id                             (number_of_flashes) uint64 149kB 0 ....
Attributes:
    title:                    Lightning Mapping Array Dataset, L1b events and...
    production_date:          2026-07-21 13:56:54 +00:00
    production_site:          Default
    institution:              unknown
    comment:                  
    history:                  LMA source file created  Wed Jun 28 19:17:56 20...
    references:               
    source:                   VHF Lightning Mapping Array
    event_algorithm_name:      /lma/bin/lma_analysis -d 20210504 -t 130000 -s...
    event_algorithm_version:   10.14.9R
    flash_algorithm_name:     pyxlma DBSCAN
    flash_algorithm_version:  0.1

In [4]:
ds_orig = ds

# Pick the filtering center based on how we will define the grid below.
filter_range_ctr_lon, filter_range_ctr_lat = ds_orig.network_center_longitude.data, ds_orig.network_center_latitude.data

def get_distance(lon, lat, ctr_lon, ctr_lat):
    """ Calculate distance from a given center location using pyproj"""
    proj_map = proj4.crs.CRS(proj='aeqd', lat_0=ctr_lat, lon_0=ctr_lon)
    proj_lla = proj4.crs.CRS(proj='latlong')    
    trnsf_to_map = proj4.Transformer.from_crs(proj_lla, proj_map)
    trnsf_from_map = proj4.Transformer.from_crs(proj_map, proj_lla)
    dist_x, dist_y = trnsf_to_map.transform(lon, lat)
    return dist_x, dist_y


fl_init_dist_x, fl_init_dist_y = get_distance(ds_orig.flash_init_longitude.data, ds_orig.flash_init_latitude.data,
                                              ctr_lon=filter_range_ctr_lon, ctr_lat=filter_range_ctr_lat)
fl_init_range = np.sqrt(fl_init_dist_x*fl_init_dist_x + fl_init_dist_y*fl_init_dist_y)

# Add a new variable of the same shape as the original number of flashes,
# and tell xarray that it has the same number of elements as the existing number_of_flashes_dimension
ds_orig['flash_init_range_from_network_center'] = xr.DataArray(fl_init_range, dims='number_of_flashes')

ds = filter_flashes(ds_orig, flash_init_range_from_network_center=(0, 150.0e3), flash_event_count=(5,None))

ds

<xarray.Dataset> Size: 154MB
Dimensions:                               (number_of_stations: 14,
                                           number_of_events: 1837120,
                                           number_of_flashes: 15943)
Coordinates:
  * number_of_flashes                     (number_of_flashes) uint64 128kB 0 ...
Dimensions without coordinates: number_of_stations, number_of_events
Data variables: (12/43)
    network_center_latitude               float64 8B 34.72
    network_center_longitude              float64 8B -86.65
    network_center_altitude               float64 8B 0.0
    station_latitude                      (number_of_stations) float32 56B 34...
    station_longitude                     (number_of_stations) float32 56B -8...
    station_altitude                      (number_of_stations) float32 56B 20...
    ...                                    ...
    event_parent_flash_id                 (number_of_events) uint64 15MB 0 .....
    event_x                               (number_of_events) float64 15MB -1....
    event_y                               (number_of_events) float64 15MB -4....
    event_z                               (number_of_events) float32 7MB 1.00...
    flash_id                              (number_of_flashes) uint64 128kB 0 ...
    flash_init_range_from_network_center  (number_of_flashes) float64 128kB 1...
Attributes:
    title:                    Lightning Mapping Array Dataset, L1b events and...
    production_date:          2026-07-21 13:56:54 +00:00
    production_site:          Default
    institution:              unknown
    comment:                  
    history:                  LMA source file created  Wed Jun 28 19:17:56 20...
    references:               
    source:                   VHF Lightning Mapping Array
    event_algorithm_name:      /lma/bin/lma_analysis -d 20210504 -t 130000 -s...
    event_algorithm_version:   10.14.9R
    flash_algorithm_name:     pyxlma DBSCAN
    flash_algorithm_version:  0.1

In [5]:
#Add a gridded dataset to the dataset

if False:
    print("Writing data")
    duration_sec = (dttuple[1]-dttuple[0]).total_seconds()
    date_fmt = "LYLOUT_%y%m%d_%H%M%S_{0:04d}_flash.nc".format(int(duration_sec))
    outfile = dttuple[0].strftime(date_fmt)

    # Compress the variables.
    comp = dict(zlib=True, complevel=5)
    encoding = {var: comp for var in ds.data_vars}
    ds.to_netcdf(outfile, encoding=encoding)

print("Setting up grid spec")
grid_dt = np.asarray(grid_time_delta_sec, dtype='m8[s]')
grid_t0 = np.asarray(dttuple[0]).astype('datetime64[ns]')
grid_t1 = np.asarray(dttuple[1]).astype('datetime64[ns]')
time_range = (grid_t0, grid_t1+grid_dt, grid_dt)

# Change the dictionaries below to a consistent set of coordinates
# and adjust grid_spatial_coords in the call to events_to_grid to
# change what is gridded (various time series of 1D, 2D, 3D grids)

if latlon_grid:
    # center = 29.7600000, -95.3700000
    lat_range = (filter_range_ctr_lat-1.5, filter_range_ctr_lat+1.5, resolution_m/110.0e3)
    lon_range = (filter_range_ctr_lon-1.5, filter_range_ctr_lon+1.5, resolution_m/110.0e3)
    alt_range = (0, 15e3, 1.0e3)


    grid_edge_ranges ={
        'grid_latitude_edge':lat_range,
        'grid_longitude_edge':lon_range,
    #     'grid_altitude_edge':alt_range,
        'grid_time_edge':time_range,
    }
    grid_center_names ={
        'grid_latitude_edge':'grid_latitude',
        'grid_longitude_edge':'grid_longitude',
    #     'grid_altitude_edge':'grid_altitude',
        'grid_time_edge':'grid_time',
    }

    event_coord_names = {
        'event_latitude':'grid_latitude_edge',
        'event_longitude':'grid_longitude_edge',
    #     'event_altitude':'grid_altitude_edge',
        'event_time':'grid_time_edge',
    }

    flash_ctr_names = {
        'flash_init_latitude':'grid_latitude_edge',
        'flash_init_longitude':'grid_longitude_edge',
    #     'flash_init_altitude':'grid_altitude_edge',
        'flash_time_start':'grid_time_edge',
    }
    flash_init_names = {
        'flash_center_latitude':'grid_latitude_edge',
        'flash_center_longitude':'grid_longitude_edge',
    #     'flash_center_altitude':'grid_altitude_edge',
        'flash_time_start':'grid_time_edge',
    }
else:
    # Use meters in a map projection coordinate
    
    prj_map = proj4.crs.CRS(proj='aeqd', ellps='WGS84',
                         lat_0=filter_range_ctr_lat, lon_0=filter_range_ctr_lon)
    prj_lla = proj4.crs.CRS(proj='latlong', ellps='WGS84')
    
    # prj_lla, prj_map, x_edge, y_edge = get_coord_proj()
    # prj_dx = x_edge[1] - x_edge[0]
    # prj_dy = y_edge[1] - y_edge[0]
    # lma_prj_xratio = resolution_m/prj_dx
    # lma_prj_yratio = resolution_m/prj_dy
    trnsf_to_map = proj4.Transformer.from_crs(prj_lla, prj_map)
    trnsf_from_map = proj4.Transformer.from_crs(prj_map, prj_lla)
    lmax, lmay = trnsf_to_map.transform(#prj_lla, prj_map,
                                 ds.event_longitude.data,
                                 ds.event_latitude.data)
    lma_initx, lma_inity = trnsf_to_map.transform(#prj_lla, prj_map,
                                 ds.flash_init_longitude.data,
                                 ds.flash_init_latitude.data)
    lma_ctrx, lma_ctry = trnsf_to_map.transform(#prj_lla, prj_map,
                                 ds.flash_center_longitude.data,
                                 ds.flash_center_latitude.data)
    ds['event_x'] = xr.DataArray(lmax, dims='number_of_events')
    ds['event_y'] = xr.DataArray(lmay, dims='number_of_events')
    ds['flash_init_x'] = xr.DataArray(lma_initx, dims='number_of_flashes')
    ds['flash_init_y'] = xr.DataArray(lma_inity, dims='number_of_flashes')
    ds['flash_ctr_x'] = xr.DataArray(lma_ctrx, dims='number_of_flashes')
    ds['flash_ctr_y'] = xr.DataArray(lma_ctry, dims='number_of_flashes')

    grid_edge_ranges ={
        'grid_x_edge':(-150e3,150e3+.001,resolution_m),
        'grid_y_edge':(-150e3,150e3+.001,resolution_m),
    #     'grid_altitude_edge':alt_range,
        'grid_time_edge':time_range,
    }
    grid_center_names ={
        'grid_x_edge':'grid_x',
        'grid_y_edge':'grid_y',
    #     'grid_altitude_edge':'grid_altitude',
        'grid_time_edge':'grid_time',
    }

    event_coord_names = {
        'event_x':'grid_x_edge',
        'event_y':'grid_y_edge',
    #     'event_altitude':'grid_altitude_edge',
        'event_time':'grid_time_edge',
    }

    flash_ctr_names = {
        'flash_init_x':'grid_x_edge',
        'flash_init_y':'grid_y_edge',
    #     'flash_init_altitude':'grid_altitude_edge',
        'flash_time_start':'grid_time_edge',
    }
    flash_init_names = {
        'flash_ctr_x':'grid_x_edge',
        'flash_ctr_y':'grid_y_edge',
    #     'flash_center_altitude':'grid_altitude_edge',
        'flash_time_start':'grid_time_edge',
    }


print("Creating regular grid")
grid_ds = create_regular_grid(grid_edge_ranges, grid_center_names)
if latlon_grid:
    pass
else:
    ctrx, ctry = np.meshgrid(grid_ds.grid_x, grid_ds.grid_y)
    hlon, hlat = trnsf_from_map.transform(ctrx, ctry)
    # Add lon lat to the dataset, too.
    ds['lon'] = xr.DataArray(hlon, dims=['grid_y', 'grid_x'],
                    attrs={'standard_name':'longitude'})
    ds['lat'] = xr.DataArray(hlat, dims=['grid_y', 'grid_x'],
                    attrs={'standard_name':'latitude'})

print("Finding grid position for flashes")
pixel_id_var = 'event_pixel_id'
ds_ev = assign_regular_bins(grid_ds, ds, event_coord_names,
    pixel_id_var=pixel_id_var, append_indices=True)
# ds_flctr = assign_regular_bins(grid_ds, ds, flash_ctr_names,
#     pixel_id_var='flash_ctr_pixel_id', append_indices=True)
# flctr_gb = ds.groupby('flash_ctr_pixel_id')
# ds_flini = assign_regular_bins(grid_ds, ds, flash_init_names,
#     pixel_id_var='flash_init_pixel_id', append_indices=True)
# flini_gb = ds.groupby('flash_init_pixel_id')

# print('===== ev_gb')
# for event_pixel_id, dsegb in ev_gb:
#     print(dsegb)
#     break
# print('===== flctr_gb')
# for event_pixel_id, dsfgb in flctr_gb:
#     print(dsfgb)
#     break

print("Gridding data")
if latlon_grid:
    grid_spatial_coords=['grid_time', None, 'grid_latitude', 'grid_longitude']
    event_spatial_vars = ('event_altitude', 'event_latitude', 'event_longitude')
else:
    grid_spatial_coords=['grid_time', None, 'grid_y', 'grid_x']
    event_spatial_vars = ('event_altitude', 'event_y', 'event_x')

# print(ds_ev)
# print(grid_ds)
grid_ds = events_to_grid(ds_ev, grid_ds, min_points_per_flash=3,
                         pixel_id_var=pixel_id_var,
                         event_spatial_vars=event_spatial_vars,
                         grid_spatial_coords=grid_spatial_coords)

# Let's combine the flash and event data with the gridded data into one giant data structure.
both_ds = xr.combine_by_coords((grid_ds, ds))
print(both_ds)

Setting up grid spec
Creating regular grid
Finding grid position for flashes
Gridding data
<xarray.Dataset> Size: 159MB
Dimensions:                               (grid_x_edge: 76, grid_x: 75,
                                           grid_y_edge: 76, grid_y: 75,
                                           grid_time_edge: 15, grid_time: 14,
                                           number_of_stations: 14,
                                           number_of_events: 1837120,
                                           number_of_flashes: 15943)
Coordinates:
  * grid_x_edge                           (grid_x_edge) float64 608B -1.5e+05...
  * grid_x                                (grid_x) float64 600B -1.48e+05 ......
  * grid_y_edge                           (grid_y_edge) float64 608B -1.5e+05...
  * grid_y                                (grid_y) float64 600B -1.48e+05 ......
  * grid_time_edge                        (grid_time_edge) datetime64[ns] 120B ...
  * grid_time                   

In [6]:
#Write the data to a netCDF for later use

if True:
    print("Writing data")
    duration_sec = (dttuple[1]-dttuple[0]).total_seconds()
    if latlon_grid:
        date_fmt = "LYLOUT_%y%m%d_%H%M%S_{0:04d}_grid.nc".format(int(duration_sec))
    else:
        date_fmt = "LYLOUT_%y%m%d_%H%M%S_{0:04d}_map{1:d}m.nc".format(
                        int(duration_sec), resolution_m)
    outfile = dttuple[0].strftime(date_fmt)
    print(outfile)
    comp = dict(zlib=True, complevel=5)
    encoding = {var: comp for var in both_ds.data_vars}
    both_ds.to_netcdf(outfile, encoding=encoding)

Writing data
LYLOUT_210504_130000_4200_map4000m.nc


# Run Adapt

After the LMA data is fully processed, Adapt can be run to get statistics on individual storm cells. 

The current setup uses KHTX for North Alabama and can be changed to fit whatever radar works best for the desired dataset. Day range can also be modified. 

In [8]:
%%bash

#This step takes about 30 minutes to process the radar data

adapt run-nexrad --radar KHTX --mode historical --start-time "2021-05-04 13:00:00" --end-time "2021-05-04 16:00:00" --base-dir /*/*/Adapt_pyxlma/Adapt_Output --rerun

2026-07-21 09:00:16 - adapt.persistence.catalog - INFO - RadarCatalog initialized for KHTX at /home/blamsma/Adapt_pyxlma/Adapt_Output/KHTX/catalog.db
2026-07-21 09:00:16 - adapt.runtime.orchestrator - INFO - Pipeline started | run=2026JUL21-0900-KHTX radar=KHTX mode=HISTORICAL
2026-07-21 09:00:18 - adapt.visualization.plotter - INFO - RadarPlotter initialized (format=png, dpi=200)
2026-07-21 09:00:18 - adapt.visualization.plotter - INFO - PlotConsumer initialized (poll_interval=2.0s, output_dir=/home/blamsma/Adapt_pyxlma/Adapt_Output/KHTX/plots)
2026-07-21 09:00:18 - adapt.visualization.plotter - INFO - PlotConsumer started, polling for new analysis artifacts...
2026-07-21 09:00:18 - adapt.modules.acquisition.module - INFO - Starting Downloader-KHTX in historical mode
2026-07-21 09:00:18 - adapt.modules.acquisition.module - INFO - Historical: 2021-05-04 13:00:00 to 2021-05-04 16:00:00
2026-07-21 09:00:20 - adapt.runtime.processor - INFO - Enabled modules: [ingest, detection, projection


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly supported
## by the U.S. Department of Energy Office of Science as part of
## the Atmospheric Radiation Measurement (ARM) User Facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119

Cleaning radar output directory: /home/blamsma/Adapt_pyxlma/Adapt_Output/KHTX
Radar output cleaned
Downloaded KHTX20210504_130449_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_131045_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_131636_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_132238_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_132839_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_133307_V06
1 out of 1 files downloaded...0 errors
Downloaded KHTX20210504_133731_V06
1 out of 

# Adapt Postprocess

Similar to running adapt, the postprocess module takes a couple of inputs for where the adapt output (repository) and the LMA output (input-dir) to assign LMA characteristics to individual storm cells. 

In [10]:
%%bash

adapt postprocess --repository /*/*/Adapt_pyxlma/Adapt_Output/ --module xlma_stat --input-dir /*/*/Adapt_pyxlma/

INFO:adapt.persistence.catalog:RadarCatalog initialized for KHTX at /home/blamsma/Adapt_pyxlma/Adapt_Output/KHTX/catalog.db
INFO:adapt.runtime.postprocessor:Post-processing modules: [xlma_stat]
INFO:adapt.persistence.scan_mask_reader:Skipping 6 advected minutes of 0322f6e52b9a4b50: no registration_cell_uid (first scan pair of the run has no previous tracking)
INFO:adapt.execution.nodes.xlma_stat:xlma_stat: ignoring 6 non-NetCDF file(s) in /home/blamsma/Adapt_pyxlma/
INFO:adapt.modules.xlma_stat.module:xlma_stat: excluding 2755 of 15943 flashes outside the run's minute-mask coverage [2021-05-04 13:11:00 .. 2021-05-04 14:13:00]
INFO:adapt.execution.nodes.xlma_stat:xlma_stat: 15943 flashes -> 2082 (cell, minute) rows, 616 (cell, scan) rows



## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly supported
## by the U.S. Department of Energy Office of Science as part of
## the Atmospheric Radiation Measurement (ARM) User Facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119

Continuing existing run ID: 2026JUL21-0900-KHTX
Ignoring user config file and CLI config overrides; reusing saved runtime config for this run.


In [13]:
# Change file path to match where the adapt output catalog is located 
dbfile = '/home/blamsma/Adapt_pyxlma/Adapt_Output/KHTX/catalog.db'
# Create a SQL connection to our SQLite database
con = sqlite3.connect(dbfile)

# creating cursor
cur = con.cursor()

# reading all table names
table_list = [a for a in cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'")]
conn = sqlite3.connect("/home/blamsma/adapt_output/KHTX/25-05-03catalog.db")
cell_events = pd.read_sql_query("SELECT * FROM cell_events", conn)
cell_tracks = pd.read_sql_query("SELECT * FROM cell_tracks", conn)
cell_volume_stats = pd.read_sql_query("SELECT * FROM cell_volume_stats", conn)
cells_by_scan = pd.read_sql_query("SELECT * FROM cells_by_scan", conn)
xlma_stat_minutes = pd.read_sql_query("SELECT * FROM xlma_stat_minutes", conn)
xlma_stat_minutes = xlma_stat_minutes.rename(columns={"target_scan_time": "scan_time"})
xlma_stat_scan = pd.read_sql_query("SELECT * FROM xlma_stat_scan", conn)
# here is you table list
print(table_list)

# Be sure to close the connection
con.close()

[('items',), ('progress',), ('schemas',), ('scans',), ('cells_by_scan',), ('cell_events',), ('cell_tracks',), ('cell_volume_stats',), ('module_schemas',), ('xlma_stat_minutes',), ('xlma_stat_scan',)]


In [14]:
#print out the disired dataframes and use the data for any analysis

#cell_events
#cell_tracks
#cell_volume_stats
#cells_by_scan
#xlma_stat_minutes
#xlma_stat_scan